# EXPT NO: 6 — IMPLEMENTATION OF LSTM FOR SENTIMENT ANALYSIS

**Dataset:** IMDB Movie Reviews (loaded from Hugging Face `datasets`, 50,000 labelled reviews: positive / negative)

**Aim:** To implement a Long Short-Term Memory (LSTM) network for sentiment analysis by performing text preprocessing, tokenization, sequence generation and embedding, and to classify reviews into positive / negative sentiment.

**How to run:** `Runtime → Run all`. Switch to a GPU (`Runtime → Change runtime type → T4 GPU`) for faster training; CPU also works.

In [ ]:
!pip install -q datasets
print("HARSHAVARTHAN SB 24BAD033")

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
import re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

import tensorflow as tf
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
except ImportError:                                   # newer Keras 3 builds
    from keras.src.legacy.preprocessing.text import Tokenizer
try:
    from tensorflow.keras.utils import pad_sequences
except ImportError:
    from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available     :", bool(tf.config.list_physical_devices('GPU')))

---
## PART A: DATASET PREPARATION AND TEXT PREPROCESSING

### A1. Load the IMDB dataset

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
from datasets import load_dataset

# To keep training quick in Colab we use a random subset of the 50,000 reviews.
# (Increase these numbers for higher accuracy, at the cost of longer training.)
N_TRAIN = 20000
N_TEST  = 5000

imdb = load_dataset("stanfordnlp/imdb")
train_df = imdb["train"].shuffle(seed=SEED).select(range(N_TRAIN)).to_pandas()
test_df  = imdb["test"].shuffle(seed=SEED).select(range(N_TEST)).to_pandas()

print(f"Training reviews : {len(train_df)}")
print(f"Test reviews     : {len(test_df)}")
train_df.head()

### A2. Explore the dataset — samples and class distribution

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
label_names = {0: "Negative", 1: "Positive"}

print(f"Class distribution (train):")
print(train_df["label"].map(label_names).value_counts(), "\n")

train_df["label"].map(label_names).value_counts().reindex(["Negative", "Positive"]).plot(
    kind="bar", color=["tab:red", "tab:green"], rot=0)
plt.title(f"Sentiment class distribution")
plt.ylabel("Number of reviews")
plt.show()

for lbl in [1, 0]:
    print(f"--- Sample {label_names[lbl].upper()} review ---")
    print(train_df[train_df["label"] == lbl]["text"].iloc[0][:500], "...\n")

train_df["n_words"] = train_df["text"].str.split().str.len()
print(f"Review length (words): mean={train_df['n_words'].mean():.0f}, "
      f"median={train_df['n_words'].median():.0f}, max={train_df['n_words'].max()}")

### A3. Text preprocessing (NLTK)
Steps: lowercase → remove HTML tags → remove punctuation / special characters → tokenize → remove stop words.

> We keep negation words (*not, no, nor, never…*) because removing them can flip the meaning of a sentence (e.g. *"not good"* would become *"good"*).

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
negations = {"no", "nor", "not", "don", "don't", "didn", "didn't", "doesn", "doesn't",
             "isn", "isn't", "wasn", "wasn't", "aren", "aren't", "couldn", "couldn't",
             "wouldn", "wouldn't", "shouldn", "shouldn't", "hadn", "hadn't", "hasn",
             "hasn't", "haven", "haven't", "won", "won't", "weren", "weren't", "never"}
stop_words = set(stopwords.words("english")) - negations

def preprocess(text):
    text = text.lower()                          # 1. lowercase
    text = re.sub(r"<.*?>", " ", text)           # 2. remove HTML tags like <br />
    text = re.sub(r"[^a-z\s]", " ", text)        # 3. remove punctuation, digits, special chars
    tokens = text.split()                        # 4. tokenize into words
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]  # 5. remove stop words
    return " ".join(tokens)

example = train_df["text"].iloc[0]
print("BEFORE:", example[:300])
print("AFTER :", preprocess(example)[:300])

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
train_df["clean"] = train_df["text"].apply(preprocess)
test_df["clean"]  = test_df["text"].apply(preprocess)
print(f"Preprocessing complete.")
train_df[["clean", "label"]].head()

### A4. Train / validation / test split and label encoding

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
# Labels are already numeric: 0 = Negative, 1 = Positive
X_train_text, X_val_text, y_train, y_val = train_test_split(
    train_df["clean"].values, train_df["label"].values,
    test_size=0.2, random_state=SEED, stratify=train_df["label"].values)

X_test_text = test_df["clean"].values
y_test      = test_df["label"].values

print(f"Train: {len(X_train_text)} | Validation: {len(X_val_text)} | Test: {len(X_test_text)}")

### A5. Tokenization, vocabulary and padding

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
VOCAB_SIZE = 10000     # keep the 10,000 most frequent words
MAX_LEN    = 200       # every review is padded / truncated to 200 tokens

# Vocabulary is built ONLY from the training reviews (no data leakage)
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

print(f"Unique words found in training data: {len(tokenizer.word_index)}")
print("Top 15 words in vocabulary:", list(tokenizer.word_index.items())[:15])

# Text -> integer sequences
train_seq = tokenizer.texts_to_sequences(X_train_text)
val_seq   = tokenizer.texts_to_sequences(X_val_text)
test_seq  = tokenizer.texts_to_sequences(X_test_text)

print("\nExample review (text)     :", X_train_text[0][:100], "...")
print("Example review (sequence) :", train_seq[0][:15], "...")

# Padding -> equal-length input
X_train = pad_sequences(train_seq, maxlen=MAX_LEN, padding="pre", truncating="post")
X_val   = pad_sequences(val_seq,   maxlen=MAX_LEN, padding="pre", truncating="post")
X_test  = pad_sequences(test_seq,  maxlen=MAX_LEN, padding="pre", truncating="post")

print(f"\nPadded shapes -> X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print("Padded example:", X_train[0][:25], "...")

---
## PART B: EMBEDDING GENERATION

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
EMBED_DIM = 64   # each word is represented by a 64-dimensional dense vector

print(f"Vocabulary size  : {VOCAB_SIZE}")
print(f"Embedding dim    : {EMBED_DIM}")

# Stand-alone Embedding layer to show what it produces
demo_embedding = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)

sample_batch  = X_train[:3]                    # 3 padded reviews
sample_embeds = demo_embedding(sample_batch)   # dense vectors

print(f"\nInput  shape (batch, seq_len)             : {sample_batch.shape}")
print(f"Output shape (batch, seq_len, embed_dim)  : {sample_embeds.shape}")
print(f"Embedding matrix (weights) shape          : {demo_embedding.get_weights()[0].shape}")

# How a single word is represented
word = "great"
idx  = tokenizer.word_index[word]
print(f"\nWord '{word}' -> index {idx} -> vector (first 8 of {EMBED_DIM} values):")
print(np.round(demo_embedding.get_weights()[0][idx][:8], 4))
print("\nNOTE: these vectors are randomly initialised. They become meaningful only after the "
      "model is trained (see the embedding-space analysis at the end of Part D).")

---
## PART C: IMPLEMENTING THE LSTM MODEL

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
model = models.Sequential([
    layers.Input(shape=(MAX_LEN,), name="padded_input"),                     # input layer
    layers.Embedding(VOCAB_SIZE, EMBED_DIM, name="embedding"),               # dense word vectors
    layers.LSTM(64, name="lstm"),                                            # learns sequential dependencies
    layers.Dropout(0.5, name="dropout"),                                     # reduces over-fitting
    layers.Dense(1, activation="sigmoid", name="output")                     # P(positive)
], name="LSTM_Sentiment_Model")

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

print(f"Model summary")
model.summary()

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
# Optional: model architecture diagram (needs graphviz; skipped silently if unavailable)
try:
    display(tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True))
except Exception as e:
    print("Architecture plot skipped:", type(e).__name__)

---
## PART D: MODEL TRAINING AND SENTIMENT PREDICTION

### D1. Train the model (with validation)

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

print(f"Training started...")
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=8,
                    batch_size=128,
                    callbacks=[early_stop],
                    verbose=1)

### D2. Training / validation accuracy and loss

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
hist = pd.DataFrame(history.history)
hist.index = hist.index + 1
hist.index.name = "epoch"
print(f"Metrics per epoch:")
display(hist.round(4))

print(f"\nFinal Training Accuracy   : {hist['accuracy'].iloc[-1]:.4f}")
print(f"Final Validation Accuracy : {hist['val_accuracy'].iloc[-1]:.4f}")
print(f"Final Training Loss       : {hist['loss'].iloc[-1]:.4f}")
print(f"Final Validation Loss     : {hist['val_loss'].iloc[-1]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist.index, hist["accuracy"], "o-", label="Training")
ax[0].plot(hist.index, hist["val_accuracy"], "o-", label="Validation")
ax[0].set_title(f"Accuracy vs. Epoch"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Accuracy"); ax[0].legend()

ax[1].plot(hist.index, hist["loss"], "o-", label="Training")
ax[1].plot(hist.index, hist["val_loss"], "o-", label="Validation")
ax[1].set_title(f"Loss vs. Epoch"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Loss"); ax[1].legend()
plt.tight_layout(); plt.show()

### D3. Evaluate on the test set

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\n=== Evaluation metrics (test set) ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred):.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"], digits=4))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix")
plt.show()

### D4. Predict sample reviews — predicted vs. actual

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
sample_idx = np.random.RandomState(SEED).choice(len(X_test), 8, replace=False)

rows = []
for i in sample_idx:
    rows.append({
        "Review (first 90 chars)": test_df["text"].iloc[i][:90].replace("\n", " ") + "...",
        "Actual":    label_names[y_test[i]],
        "Predicted": label_names[y_pred[i]],
        "P(positive)": round(float(y_prob[i]), 3),
        "Match": "✅" if y_test[i] == y_pred[i] else "❌",
    })
print(f"Predicted vs Actual sentiment")
display(pd.DataFrame(rows))

### D5. Try your own reviews

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
def predict_sentiment(review):
    clean = preprocess(review)
    seq   = tokenizer.texts_to_sequences([clean])
    pad   = pad_sequences(seq, maxlen=MAX_LEN, padding="pre", truncating="post")
    p     = float(model.predict(pad, verbose=0)[0][0])
    return ("Positive" if p >= 0.5 else "Negative"), p

my_reviews = [
    "This movie was absolutely fantastic! The acting was brilliant and the story kept me hooked.",
    "Terrible film. Boring plot, awful acting, and a complete waste of time.",
    "It was not good at all. I regret watching it.",
    "A heartwarming and beautifully directed masterpiece.",
]
print(f"Custom review predictions")
for r in my_reviews:
    label, p = predict_sentiment(r)
    print(f"{label:8s} (P(pos)={p:.3f})  <-  {r}")

### D6. Embedding-space analysis (after training)

In [ ]:
print("HARSHAVARTHAN SB 24BAD033")
emb_weights = model.get_layer("embedding").get_weights()[0]        # (VOCAB_SIZE, EMBED_DIM)
print(f"Trained embedding matrix shape: {emb_weights.shape}")

valid_ids = np.array([i for i in tokenizer.index_word if 1 < i < VOCAB_SIZE])   # skip padding(0) and <OOV>(1)

def most_similar(word, top_n=6):
    idx = tokenizer.word_index.get(word)
    if idx is None or idx >= VOCAB_SIZE:
        return f"'{word}' not in vocabulary"
    v = emb_weights[idx]
    cand = emb_weights[valid_ids]
    sims = cand @ v / (np.linalg.norm(cand, axis=1) * np.linalg.norm(v) + 1e-9)
    order = np.argsort(-sims)
    out = [(tokenizer.index_word[int(valid_ids[j])], round(float(sims[j]), 3))
           for j in order if int(valid_ids[j]) != idx]
    return out[:top_n]

for w in ["great", "terrible", "boring", "excellent"]:
    print(f"Words closest to '{w}':", most_similar(w))

# 2-D visualisation of selected words using PCA
from sklearn.decomposition import PCA
pos_words = ["great", "excellent", "wonderful", "amazing", "best", "love", "perfect", "beautiful"]
neg_words = ["terrible", "awful", "worst", "boring", "bad", "poor", "waste", "horrible"]
words = [w for w in pos_words + neg_words if w in tokenizer.word_index and tokenizer.word_index[w] < VOCAB_SIZE]
vecs  = np.array([emb_weights[tokenizer.word_index[w]] for w in words])
xy    = PCA(n_components=2).fit_transform(vecs)

plt.figure(figsize=(8, 6))
for (px, py), w in zip(xy, words):
    color = "tab:green" if w in pos_words else "tab:red"
    plt.scatter(px, py, c=color); plt.annotate(w, (px, py), xytext=(4, 4), textcoords="offset points")
plt.title(f"Trained word embeddings (PCA) — green: positive, red: negative")
plt.show()

---
## RESULT
The LSTM model was successfully implemented for sentiment analysis on the IMDB movie-review dataset. Text was cleaned and tokenized with NLTK / Keras, converted to padded integer sequences, mapped to dense vectors by an Embedding layer, and classified by an LSTM + Dense(sigmoid) network. Performance was evaluated using accuracy, precision, recall, F1-score and a confusion matrix.

---
## PRE-LAB ANSWERS
1. **Purpose of LSTM in NLP:** LSTMs model sequential data and remember information across long spans of text using gated memory cells, so they capture context and word order (e.g. sentiment, translation, text generation).
2. **LSTM vs Simple RNN:** A Simple RNN suffers from vanishing gradients and forgets early words in long sequences; an LSTM's gates and cell state let gradients flow over many time steps, so long-range dependencies are retained.
3. **Sentiment analysis:** Determining the emotional polarity (positive / negative / neutral) of text. Applications: analysing product reviews for brands; monitoring social-media opinion on political or public issues (also customer-support triage, stock-market mood analysis).
4. **Tokenization & padding:** Tokenization splits text into words and maps each to an integer id. Padding makes all sequences the same length so they can be batched into a fixed-size tensor.
5. **Word embedding:** A dense, low-dimensional, learnable vector for each word. Similar words get similar vectors, so the model captures semantic meaning (unlike sparse one-hot vectors) and generalises better for sentiment.

## POST-LAB ANSWERS
1. **Memory cell:** It carries information across time steps (the cell state), which the input, forget and output gates update, keep or erase — acting as the network's long-term memory.
2. **Vanishing gradient:** The cell state is updated additively and controlled by the forget gate, giving a near-constant error path, so gradients do not shrink exponentially during back-propagation through time.
3. **Embedding layer:** Converts integer word ids into trainable dense vectors that capture word meaning and are learned together with the classifier.
4. **Padding:** Gives every review equal length. The extra zeros carry no information; with `post` padding the LSTM reads pad tokens last, which dilutes the final state and can stop the model from learning. This notebook therefore uses `pre` padding (masking with `mask_zero=True` also helps).
5. **Too many epochs:** The model over-fits — training accuracy keeps rising while validation accuracy drops and validation loss rises. Use early stopping, dropout or regularisation.
6. **Precision / recall / F1:** Accuracy can be misleading with imbalanced classes. Precision shows how many predicted positives are correct, recall how many actual positives were found, and F1 balances both.
7. **Limitation of LSTM:** It processes tokens sequentially, so training is slow and hard to parallelise, and it still struggles with very long texts compared with Transformer models.